In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u

from spectral_cube import SpectralCube as sc
from tqdm.notebook import trange
import glob
import gc
import warnings
# import fitsio
warnings.filterwarnings("ignore")

In [2]:
fits_file_list = glob.glob('./South/*.fits')

group_info = [
    {
        "name": "./South/Combined/CRAFTS_RA60_80_DEC-13_2.fits",
        "files": [fits_file_list[0], fits_file_list[1], fits_file_list[8], fits_file_list[9]],
        "xlo": 60.0, "xhi": 80.0, "ylo": -13.0, "yhi": 1.99
    },
    {
        "name": "./South/Combined/CRAFTS_RA80_100_DEC-13_2.fits",
        "files": [fits_file_list[2], fits_file_list[3], fits_file_list[10], fits_file_list[11]],
        "xlo": 80.0, "xhi": 100.0, "ylo": -13.0, "yhi": 1.99
    },
    {
        "name": "./South/Combined/CRAFTS_RA100_120_DEC-13_2.fits",
        "files": [fits_file_list[4], fits_file_list[5], fits_file_list[12], fits_file_list[13]],
        "xlo": 100.0, "xhi": 120.0, "ylo": -13.0, "yhi": 1.99
    },
    {
        "name": "./South/Combined/CRAFTS_RA120_140_DEC-13_2.fits",
        "files": [fits_file_list[6], fits_file_list[7], fits_file_list[14], fits_file_list[15]],
        "xlo": 120.0, "xhi": 140.0, "ylo": -13.0, "yhi": 1.99
    }
]

In [3]:
def combine_matrices(matrix1, matrix2, matrix3, matrix4):
    bottom_row = np.concatenate((matrix4, matrix3), axis=2)
    top_row    = np.concatenate((matrix2, matrix1), axis=2)
    combined_matrix = np.concatenate((top_row, bottom_row), axis=1)
    return combined_matrix

In [ ]:
for i in trange(len(group_info)):
    group = group_info[0]
    # 读取数据
    data1 = fits.getdata(group["files"][0], memmap=True)
    data2 = fits.getdata(group["files"][1], memmap=True)
    data3 = fits.getdata(group["files"][2], memmap=True)
    data4 = fits.getdata(group["files"][3], memmap=True)
    # 拼接
    combined_data = combine_matrices(data1, data2, data3, data4)
    combined_data = combined_data << u.K
    del data1, data2, data3, data4
    gc.collect()
    # 用第2张的header为基准
    hdr = fits.getheader(group["files"][1])
    # 构建cube
    cube = sc(data=combined_data, wcs=WCS(hdr))
    cube = cube[::-1]
    print("Combined cube:")
    print(cube)
    del combined_data
    gc.collect()
    # 裁剪你需要的区域
    region_cube = cube.subcube(
        xlo=group['xlo'] * u.deg,
        xhi=group['xhi'] * u.deg,
        ylo=group['ylo'] * u.deg,
        yhi=group['yhi'] * u.deg
    )
    print("Region cube:")
    print(region_cube, "\n")
    region_cube.write(group['name'], overwrite=True)
    del cube, region_cube
    gc.collect()
    print(f"Saved {group['name']}")

  0%|          | 0/4 [00:00<?, ?it/s]

SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    6.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s
[-599897.17182941 -599695.88823468 -599494.60463995 ...  599551.76918453
  599753.05277927  599954.336374  ]
SpectralCube with shape=(5962, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s
Saved ./South/Combined/CRAFTS_RA60_80_DEC-13_2.fits
SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg    range:   